[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/himanshu231204/pytorch-mastery/blob/main/05_production_engineering/mixed_precision.ipynb)

# 01 . Mixed Precision Training

## Concept — Mixed Precision (revision notes)

**What it is**
- Running parts of training in a 16-bit float format (fp16 or bf16) while keeping numerically sensitive parts (master weights, reductions, loss) in fp32.
- In PyTorch the entry point is `torch.autocast` (per-op dtype selection) plus, for fp16 only, `torch.amp.GradScaler` (loss scaling).

**Why it matters**
- 16-bit tensors take half the memory and, on GPUs with tensor cores, matmuls/convs run several times faster.
- Halved activation memory lets you use larger batches or larger models.
- Done naively (`model.half()`), training silently diverges or stalls — autocast + loss scaling is the safe recipe.

**When to use**
- GPU training with Ampere or newer: prefer bf16 (no scaler needed). Older GPUs (V100/T4): fp16 + GradScaler.
- CPU: bf16 autocast works (useful on CPUs with AVX512-BF16/AMX); fp16 on CPU is mostly not worth it.
- Skip it for tiny models where kernel-launch overhead, not math, is the bottleneck.

**Math & intuition**
- fp32: 1 sign, 8 exponent, 23 mantissa bits. fp16: 5 exponent, 10 mantissa (max ~65504, smallest normal ~6e-5). bf16: 8 exponent, 7 mantissa (same range as fp32, but only ~2-3 significant decimal digits).
- fp16 has precision but little range -> gradients below ~6e-8 flush to zero (underflow). Loss scaling multiplies the loss by S (e.g. 65536) so gradients are shifted into range, then divides grads by S before the optimizer step.
- bf16 has fp32's range but coarse precision -> no underflow problem, so no scaler; the cost is rounding error, which is why accumulations and weight updates stay in fp32.

### 1. Dtype ranges: what each format can represent

**What this cell does (plain language):** we ask `torch.finfo` for the numeric limits of fp32, fp16 and bf16. This is the single most useful fact in mixed precision: fp16 has a tiny range, bf16 has fp32's range but fewer mantissa bits.

In [1]:
import torch, torch.nn as nn, time
torch.manual_seed(0)

for dt in (torch.float32, torch.float16, torch.bfloat16):
    fi = torch.finfo(dt)
    # max = largest finite value, tiny = smallest *normal* value,
    # eps = gap between 1.0 and the next representable number (relative precision)
    print(f"{str(dt):15s} bits={fi.bits:2d} max={fi.max:.3e} tiny={fi.tiny:.3e} eps={fi.eps:.3e}")

torch.float32   bits=32 max=3.403e+38 tiny=1.175e-38 eps=1.192e-07
torch.float16   bits=16 max=6.550e+04 tiny=6.104e-05 eps=9.766e-04
torch.bfloat16  bits=16 max=3.390e+38 tiny=1.175e-38 eps=7.812e-03


### 2. Overflow, underflow and rounding in action

**What this cell does (plain language):** we cast a few values to fp16 and bf16 and see what happens: 70000 overflows to inf in fp16 (but not bf16), 1e-8 underflows to 0 in fp16 (but not bf16), and 1.001 loses its extra digits in bf16.

In [2]:
x = torch.tensor([70000.0, 1e-8, 1.001, 3.14159265])
print("fp32:", x)
print("fp16:", x.to(torch.float16))    # 70000 -> inf (overflow), 1e-8 -> 0 (underflow)
print("bf16:", x.to(torch.bfloat16))   # keeps range, loses precision (1.001 -> 1.0)

fp32: tensor([7.0000e+04, 1.0000e-08, 1.0010e+00, 3.1416e+00])
fp16: tensor([   inf, 0.0000, 1.0010, 3.1406], dtype=torch.float16)
bf16: tensor([7.0144e+04, 1.0012e-08, 1.0000e+00, 3.1406e+00], dtype=torch.bfloat16)


### 3. fp16 gradient underflow demo and loss scaling by hand

**What this cell does (plain language):** we build a loss whose gradient is genuinely tiny (about 1e-9). In fp16 that gradient is flushed to zero, so the layer would never learn. Multiplying the loss by a scale factor first (the idea behind `GradScaler`) shifts the gradient into fp16's representable range, and we divide it back out afterwards.

In [3]:
w = torch.tensor([1.0], dtype=torch.float16, requires_grad=True)

def tiny_loss(w, scale=1.0):
    # d(loss)/dw = 1e-9 mathematically, far below fp16's smallest subnormal (~6e-8)
    return (w * torch.tensor(1e-9, dtype=torch.float16)).sum() * scale

# a) no scaling: the *constant* 1e-9 itself is already 0 in fp16, so the grad is 0
tiny_loss(w).backward()
print("grad without scaling:", w.grad.item())

# b) the realistic situation: grad arrives as a tiny fp16 number from upstream.
#    Emulate: upstream gradient 1e-9 in fp32 -> cast to fp16 -> 0
g_true = torch.tensor(1e-9)
print("1e-9 as fp16        :", g_true.half().item())
S = 2.0 ** 16                              # loss scale (a power of two keeps it exact)
g_scaled = (g_true * S).half()             # scaling the loss scales every gradient by S
print("scaled 1e-9*65536 as fp16:", g_scaled.item(), "(representable!)")
print("unscaled in fp32    :", g_scaled.float().item() / S, "(matches 1e-9)")

grad without scaling: 0.0
1e-9 as fp16        : 0.0
scaled 1e-9*65536 as fp16: 6.556510925292969e-05 (representable!)
unscaled in fp32    : 1.000444171950221e-09 (matches 1e-9)


### 4. autocast on CPU with bfloat16

**What this cell does (plain language):** we run the same Linear layer in plain fp32 and under `torch.autocast('cpu', dtype=torch.bfloat16)`. We print the output dtype and the difference. Autocast keeps the weights in fp32 and casts them per-op; matmul-like ops run in bf16; the weights are never permanently converted.

In [4]:
torch.manual_seed(0)
lin = nn.Linear(64, 64)
x = torch.randn(32, 64)

y32 = lin(x)
with torch.autocast(device_type="cpu", dtype=torch.bfloat16):
    ybf = lin(x)
    print("weight dtype inside autocast  :", lin.weight.dtype)   # still fp32 (master copy)
    print("output dtype of Linear (matmul):", ybf.dtype)          # bf16
    s = torch.softmax(ybf, dim=-1)
    print("output dtype of softmax        :", s.dtype)            # follows its (bf16) input on CPU
print("max abs diff fp32 vs bf16 output:", (y32 - ybf.float()).abs().max().item())

weight dtype inside autocast  : torch.float32
output dtype of Linear (matmul): torch.bfloat16
output dtype of softmax        : torch.bfloat16
max abs diff fp32 vs bf16 output: 0.0064237117767333984


### 5. Which ops autocast changes: a dtype probe

**What this cell does (plain language):** we call a handful of different ops inside autocast and print the result dtype for each. This shows the policy: matmul/linear/conv go low precision, while losses and some other risky ops are promoted back to fp32. The exact lists differ between CUDA and CPU (on CPU, softmax and layer_norm follow their input dtype), so always probe rather than assume.

In [5]:
a = torch.randn(8, 8); b = torch.randn(8, 8)
with torch.autocast("cpu", dtype=torch.bfloat16):
    probes = {
        "matmul":       a @ b,
        "add (fp32+fp32)": a + b,
        "softmax(bf16 in)":   torch.softmax(a @ b, -1),
        "layer_norm(bf16 in)": torch.nn.functional.layer_norm(a @ b, (8,)),
        "mse_loss(bf16 in)":  torch.nn.functional.mse_loss(a @ b, b),
        "exp(fp32 in)":       torch.exp(a),
    }
for name, t in probes.items():
    print(f"{name:20s} -> {t.dtype}")

matmul               -> torch.bfloat16
add (fp32+fp32)      -> torch.float32
softmax(bf16 in)     -> torch.bfloat16
layer_norm(bf16 in)  -> torch.bfloat16
mse_loss(bf16 in)    -> torch.float32
exp(fp32 in)         -> torch.float32


### 6. A full training step with autocast (bf16 -> no GradScaler)

**What this cell does (plain language):** we train a tiny MLP on synthetic regression data using the standard mixed-precision recipe for bf16: forward + loss inside `autocast`, backward and optimizer step OUTSIDE it. Because bf16 has fp32's exponent range, no scaler is needed. We compare the final loss with a pure fp32 run.

In [6]:
def make_data(n=512, d=16):
    g = torch.Generator().manual_seed(1)
    X = torch.randn(n, d, generator=g)
    w = torch.randn(d, 1, generator=g)
    return X, X @ w + 0.01 * torch.randn(n, 1, generator=g)

def train(use_amp, steps=150):
    torch.manual_seed(0)
    X, Y = make_data()
    model = nn.Sequential(nn.Linear(16, 64), nn.ReLU(), nn.Linear(64, 1))
    opt = torch.optim.AdamW(model.parameters(), lr=1e-2)
    for _ in range(steps):
        opt.zero_grad()
        with torch.autocast("cpu", dtype=torch.bfloat16, enabled=use_amp):
            loss = nn.functional.mse_loss(model(X), Y)   # forward + loss under autocast
        loss.backward()                                  # backward OUTSIDE autocast
        opt.step()
    return loss.item(), next(model.parameters()).dtype

for amp in (False, True):
    l, dt = train(amp)
    print(f"autocast={amp!s:5}  final loss={l:.5f}  param dtype={dt}")

autocast=False  final loss=0.01196  param dtype=torch.float32


autocast=True   final loss=0.01212  param dtype=torch.float32


### 7. GradScaler concepts: scale, unscale, inf check, skip step

**What this cell does (plain language):** `GradScaler` is mainly for fp16 on GPU, but we can write its core logic by hand on CPU to see exactly what it does: scale the loss, backprop, check the gradients for inf/nan, skip the step and halve the scale if found, otherwise unscale and step (and grow the scale after a streak of good steps). We simulate an overflow by injecting an inf gradient.

In [7]:
class MiniScaler:
    # Educational re-implementation of torch.amp.GradScaler's logic
    def __init__(self, init_scale=2.0**10, growth=2.0, backoff=0.5, interval=3):
        self.scale, self.growth, self.backoff, self.interval, self.good = init_scale, growth, backoff, interval, 0
    def step(self, opt, params, loss, inject_inf=False):
        (loss * self.scale).backward()                       # 1. scale loss -> big grads
        if inject_inf:
            params[0].grad.view(-1)[0] = float("inf")        # pretend fp16 overflowed
        found_inf = any(not torch.isfinite(p.grad).all() for p in params)
        if found_inf:                                        # 2. bad step: skip + shrink scale
            self.scale *= self.backoff; self.good = 0
            opt.zero_grad(); return "skipped"
        for p in params: p.grad /= self.scale                # 3. unscale to true gradients
        opt.step(); opt.zero_grad(); self.good += 1          # 4. real update
        if self.good % self.interval == 0: self.scale *= self.growth
        return "stepped"

torch.manual_seed(0)
m = nn.Linear(4, 1); params = list(m.parameters())
opt = torch.optim.SGD(params, lr=0.1); sc = MiniScaler()
x, y = torch.randn(16, 4), torch.randn(16, 1)
for i in range(7):
    loss = nn.functional.mse_loss(m(x), y)
    r = sc.step(opt, params, loss, inject_inf=(i == 3))
    print(f"iter {i}: {r:8s} scale={sc.scale:g}")

iter 0: stepped  scale=1024
iter 1: stepped  scale=1024
iter 2: stepped  scale=2048
iter 3: skipped  scale=1024
iter 4: stepped  scale=1024
iter 5: stepped  scale=1024
iter 6: stepped  scale=2048


### 8. The real torch.amp.GradScaler API (guarded for CPU)

**What this cell does (plain language):** we show the canonical fp16 recipe with the real `GradScaler`. It only does real work on CUDA, so we construct it with `enabled=torch.cuda.is_available()`; on this CPU machine it is a harmless no-op and the loop is just ordinary fp32 training. Copy this pattern for GPU fp16 training.

In [8]:
use_cuda = torch.cuda.is_available()
device = "cuda" if use_cuda else "cpu"
amp_dtype = torch.float16 if use_cuda else torch.bfloat16
scaler = torch.amp.GradScaler(device, enabled=use_cuda)     # no-op when disabled

torch.manual_seed(0)
model = nn.Linear(8, 1).to(device)
opt = torch.optim.SGD(model.parameters(), lr=0.05)
X, Y = torch.randn(64, 8, device=device), torch.randn(64, 1, device=device)

for step in range(5):
    opt.zero_grad(set_to_none=True)
    with torch.autocast(device, dtype=amp_dtype):
        loss = nn.functional.mse_loss(model(X).float(), Y)
    scaler.scale(loss).backward()          # scaled backward
    scaler.unscale_(opt)                   # optional: unscale so we can clip true grads
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(opt)                       # skips opt.step() if inf/nan grads were found
    scaler.update()                        # grows/shrinks the scale factor
print("scaler enabled:", scaler.is_enabled(), "| device:", device, "| final loss:", round(loss.item(), 4))

scaler enabled: False | device: cpu | final loss: 0.8919


### 9. Speed and memory: measure, don't assume

**What this cell does (plain language):** we time a bigger matmul-heavy forward pass in fp32 vs bf16 autocast on CPU and compare the memory footprint of the tensors. Memory halving is guaranteed; speed-up depends entirely on whether your CPU has native bf16 support (on many CPUs bf16 is actually SLOWER), which is exactly why you must measure.

In [9]:
torch.manual_seed(0)
big = nn.Sequential(nn.Linear(512, 512), nn.ReLU(), nn.Linear(512, 512)).eval()
xb = torch.randn(256, 512)

def bench(fn, n=20):
    fn()                                  # warm-up
    t = time.perf_counter()
    for _ in range(n): fn()
    return (time.perf_counter() - t) / n * 1e3

with torch.no_grad():
    t32 = bench(lambda: big(xb))
    def f16():
        with torch.autocast("cpu", dtype=torch.bfloat16): return big(xb)
    tbf = bench(f16)
print(f"fp32 forward: {t32:.2f} ms | bf16 autocast forward: {tbf:.2f} ms (CPU-dependent!)")
print("activation bytes fp32:", xb.numel() * 4, "| bf16:", xb.to(torch.bfloat16).numel() * 2)

fp32 forward: 1.14 ms | bf16 autocast forward: 0.62 ms (CPU-dependent!)
activation bytes fp32: 524288 | bf16: 262144


### 10. Numerical gotchas: accumulation and master weights

**What this cell does (plain language):** we add 0.0001 to a value of 1.0 a thousand times, once accumulating in bf16 and once in fp32. In bf16 the increment is smaller than the spacing between representable numbers near 1.0, so the sum never moves. This is why optimizers keep fp32 master weights and why you never call `model.bfloat16()` and train naively.

In [10]:
acc_bf, acc_32 = torch.tensor(1.0, dtype=torch.bfloat16), torch.tensor(1.0)
for _ in range(1000):
    acc_bf = acc_bf + torch.tensor(1e-4, dtype=torch.bfloat16)
    acc_32 = acc_32 + 1e-4
print("bf16 accumulation:", acc_bf.item(), "(expected ~1.1)")
print("fp32 accumulation:", round(acc_32.item(), 4))
print("bf16 spacing near 1.0:", torch.finfo(torch.bfloat16).eps)

bf16 accumulation: 1.0 (expected ~1.1)
fp32 accumulation: 1.1
bf16 spacing near 1.0: 0.0078125


## Common bugs

- **Calling `model.half()` / `model.bfloat16()` and training with a normal optimizer.** Fix: Keep fp32 master weights and use `torch.autocast`; weight updates smaller than the fp16/bf16 spacing are silently lost.
- **Using fp16 without a `GradScaler`.** Fix: Gradients underflow to zero and the model stalls; use `GradScaler` (or switch to bf16 which needs none).
- **Putting `backward()` or `optimizer.step()` inside the `autocast` block.** Fix: Only forward + loss go inside autocast; backward runs automatically with the matching dtypes.
- **Calling `scaler.step(opt)` after clipping without `scaler.unscale_(opt)`.** Fix: Unscale first, otherwise you clip scaled gradients against an unscaled threshold.
- **Computing the loss or softmax manually in fp16.** Fix: Cast logits `.float()` before the loss, or use the built-in loss functions that autocast keeps in fp32.
- **Assuming mixed precision is always faster.** Fix: Profile it: on CPUs without bf16 hardware or for tiny models it can be slower; benefit needs tensor cores / AMX.
- **Comparing fp32 and bf16 results with tight tolerances in tests.** Fix: Use tolerances like `atol=1e-2` for bf16; it has only ~3 significant digits.
- **Forgetting `enabled=` flags so code can't switch AMP off for debugging.** Fix: Pass `enabled=use_amp` to both autocast and GradScaler; compare against fp32 when debugging NaNs.

## Exercises

**Exercise 1.** Use `torch.finfo` to find the smallest positive subnormal fp16 value (hint: it is `tiny * eps`) and check it matches the underflow threshold seen in section 3.

In [11]:
# Your attempt for Exercise 1 here

**Solution 1**

The smallest subnormal is `tiny * eps` ~ 6e-8; anything much smaller rounds to 0.

In [12]:
# Solution 1
import torch
fi = torch.finfo(torch.float16)
smallest_subnormal = fi.tiny * fi.eps
print(smallest_subnormal)                       # ~5.96e-08
print(torch.tensor(smallest_subnormal).half().item(), torch.tensor(smallest_subnormal / 4).half().item())

5.960464477539063e-08
5.960464477539063e-08 0.0


**Exercise 2.** Write a loop that finds the largest power of two `S` such that `1.0 * S` is still finite in fp16.

In [13]:
# Your attempt for Exercise 2 here

**Solution 2**

In [14]:
# Solution 2
import torch
S = 1.0
while torch.isfinite(torch.tensor(S * 2).half()): S *= 2
print(S)   # 32768.0 (2**15); 2**16 = 65536 > fp16 max 65504 overflows

32768.0


**Exercise 3.** Run the autocast dtype probe for `torch.bmm` and `torch.nn.functional.relu` and report the output dtypes.

In [15]:
# Your attempt for Exercise 3 here

**Solution 3**

bmm is on autocast's low-precision list; relu has no autocast rule so it keeps the input dtype.

In [16]:
# Solution 3
import torch
a = torch.randn(2, 4, 4)
with torch.autocast("cpu", dtype=torch.bfloat16):
    print("bmm :", torch.bmm(a, a).dtype)
    print("relu:", torch.relu(a).dtype)   # follows its input (fp32)

bmm : torch.bfloat16
relu: torch.float32


**Exercise 4.** Show that a bf16 round trip loses precision: compute the relative error of `x.bfloat16().float()` vs x for random normal x, and compare with fp16.

In [17]:
# Your attempt for Exercise 4 here

**Solution 4**

fp16 has 10 mantissa bits, bf16 only 7, so bf16 is about 8x coarser.

In [18]:
# Solution 4
import torch
x = torch.randn(10000)
for dt in (torch.bfloat16, torch.float16):
    err = ((x.to(dt).float() - x).abs() / x.abs().clamp_min(1e-6)).mean().item()
    print(dt, f"mean relative error {err:.2e}")   # bf16 about 8x worse than fp16

torch.bfloat16 mean relative error 1.42e-03
torch.float16 mean relative error 1.77e-04


**Exercise 5.** Extend `MiniScaler` so that the scale never drops below 1.0. Test it by injecting infs on 3 consecutive iterations starting from `init_scale=2.0`.

In [19]:
# Your attempt for Exercise 5 here

**Solution 5**

In [20]:
# Solution 5
class MiniScaler2:
    def __init__(self, init_scale=2.0): self.scale = init_scale
    def on_overflow(self):
        self.scale = max(1.0, self.scale * 0.5)
s = MiniScaler2(2.0)
for i in range(3):
    s.on_overflow(); print(i, s.scale)   # 1.0, 1.0, 1.0

0 1.0
1 1.0
2 1.0
